In [3]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [5]:
df = pd.read_csv("olist_order_reviews_dataset.csv")
df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [6]:
df.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

In [7]:
df = df[['review_comment_message']]

df = df.dropna()

df.head()

,review_comment_message
3,Recebi bem antes do prazo estipulado.
4,Parabéns lojas lannister adorei comprar pela I...
9,aparelho eficiente. no site a marca do aparelh...
12,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
15,"Vendedor confiável, produto ok e entrega antes..."


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]','',text)
    return text

df['clean_review'] = df['review_comment_message'].apply(clean_text)

df.head()

,review_comment_message,clean_review
3,Recebi bem antes do prazo estipulado.,recebi bem antes do prazo estipulado
4,Parabéns lojas lannister adorei comprar pela I...,parabns lojas lannister adorei comprar pela in...
9,aparelho eficiente. no site a marca do aparelh...,aparelho eficiente no site a marca do aparelho...
12,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",mas um pouco travandopelo valor ta boa
15,"Vendedor confiável, produto ok e entrega antes...",vendedor confivel produto ok e entrega antes d...


In [14]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
df['tokens'] = df['clean_review'].apply(word_tokenize)
df[['clean_review','tokens']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,clean_review,tokens
3,recebi bem antes do prazo estipulado,"[recebi, bem, antes, do, prazo, estipulado]"
4,parabns lojas lannister adorei comprar pela in...,"[parabns, lojas, lannister, adorei, comprar, p..."
9,aparelho eficiente no site a marca do aparelho...,"[aparelho, eficiente, no, site, a, marca, do, ..."
12,mas um pouco travandopelo valor ta boa,"[mas, um, pouco, travandopelo, valor, ta, boa]"
15,vendedor confivel produto ok e entrega antes d...,"[vendedor, confivel, produto, ok, e, entrega, ..."


In [15]:
lemmatizer = WordNetLemmatizer()

df['lemmatized'] = df['tokens'].apply(
    lambda words:[lemmatizer.lemmatize(word) for word in words]
)

df[['tokens','lemmatized']].head()

,tokens,lemmatized
3,"[recebi, bem, antes, do, prazo, estipulado]","[recebi, bem, ante, do, prazo, estipulado]"
4,"[parabns, lojas, lannister, adorei, comprar, p...","[parabns, lojas, lannister, adorei, comprar, p..."
9,"[aparelho, eficiente, no, site, a, marca, do, ...","[aparelho, eficiente, no, site, a, marca, do, ..."
12,"[mas, um, pouco, travandopelo, valor, ta, boa]","[ma, um, pouco, travandopelo, valor, ta, boa]"
15,"[vendedor, confivel, produto, ok, e, entrega, ...","[vendedor, confivel, produto, ok, e, entrega, ..."


In [17]:
nltk.download('averaged_perceptron_tagger_eng')
df['pos_tags'] = df['lemmatized'].apply(nltk.pos_tag)

df[['lemmatized','pos_tags']].head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


,lemmatized,pos_tags
3,"[recebi, bem, ante, do, prazo, estipulado]","[(recebi, NN), (bem, NN), (ante, NN), (do, VBP..."
4,"[parabns, lojas, lannister, adorei, comprar, p...","[(parabns, NN), (lojas, NN), (lannister, NN), ..."
9,"[aparelho, eficiente, no, site, a, marca, do, ...","[(aparelho, NN), (eficiente, VBZ), (no, DT), (..."
12,"[ma, um, pouco, travandopelo, valor, ta, boa]","[(ma, NN), (um, JJ), (pouco, NN), (travandopel..."
15,"[vendedor, confivel, produto, ok, e, entrega, ...","[(vendedor, NN), (confivel, NN), (produto, NN)..."


In [18]:
tfidf = TfidfVectorizer(max_features=500)

X_tfidf = tfidf.fit_transform(df['clean_review'])

print(X_tfidf.shape)

(40977, 500)


In [24]:
!pip install gensim
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print(model.wv['good'])

[-0.00471371  0.01156205 -0.0022325   0.00646649  0.01233385 -0.02187519
  0.01457819  0.04881913 -0.00291244 -0.02681968 -0.01369534 -0.01751444
  0.01119205  0.0052497   0.02019661 -0.00780695  0.00018555 -0.01351847
  0.0115407  -0.04756159 -0.00388196 -0.00311747  0.02888714 -0.00617962
 -0.00656973 -0.00014906 -0.01579889 -0.00116114 -0.01488682  0.02060265
  0.03256771  0.01128991  0.00191229 -0.00145578  0.00490707  0.01188415
  0.00596056 -0.01275668 -0.00298172 -0.01797355  0.01135927 -0.01402588
 -0.01134661 -0.00795247  0.00826744 -0.0144671  -0.01399141 -0.00260328
  0.01535541  0.0005693   0.01348031  0.00178593 -0.00185568  0.00967123
 -0.00340811  0.00937921  0.00072621  0.00796789 -0.00866378  0.00372573
  0.00174231  0.00809577 -0.003526   -0.01114173 -0.01653599  0.02308431
  0.03013127  0.00399503 -0.02902409  0.00993994 -0.01448329  0.01078122
  0.01145012  0.01328362  0.02372781 -0.00456272 -0.01114462 -0.02134855
 -0.03854373  0.00869973  0.00422638 -0.00310886 -0

In [20]:
!pip install -U spacy

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [33]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Take the first non-empty review
text = df["clean_review"].iloc[0]

doc = nlp(text)

print("Review:", text)
print("\nNamed Entities:")

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Review: recebi bem antes do prazo estipulado

Named Entities:


In [28]:
from textblob import TextBlob

df['polarity'] = df['clean_review'].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

def sentiment(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['polarity'].apply(sentiment)

df[['clean_review','sentiment']].head()

,clean_review,sentiment
3,recebi bem antes do prazo estipulado,Neutral
4,parabns lojas lannister adorei comprar pela in...,Neutral
9,aparelho eficiente no site a marca do aparelho...,Neutral
12,mas um pouco travandopelo valor ta boa,Neutral
15,vendedor confivel produto ok e entrega antes d...,Positive


In [29]:
df.to_csv("NLP_Output.csv", index=False)

print("Project Completed Successfully!")

Project Completed Successfully!
